In [1]:
import json
with open("data/reviews.json", encoding="utf-8") as file:
    reviews = json.load(file)

texts = [review["text"] for review in reviews]
len(texts)

12

In [2]:
from pydantic import BaseModel, Field
from typing import Literal

class Sentiment(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"] = Field(
        description="Overall sentiment of the review. Use neutral if the review is mixed"
    )

In [3]:
from pydantic_ai import Agent
from model import nvidia_model

model = nvidia_model("nvidia/nemotron-3.5-super-vl-120b-a12b")

sentiment_agent = Agent(
    model=model,
    instructions="You classify product reviews as positive, negative or neutral"
)

async def get_sentiment(text: str) -> str:
    result = await sentiment_agent.run(text, output_type=Sentiment)
    return result.output.sentiment

In [4]:
predictions = []
for text in texts:
    prediction = await get_sentiment(text)
    predictions.append(prediction)

predictions

['neutral',
 'negative',
 'neutral',
 'positive',
 'negative',
 'neutral',
 'positive',
 'negative',
 'neutral',
 'positive',
 'negative',
 'positive']

In [5]:
import pandas as pd

df = pd.DataFrame(reviews)
df["predicted"] = predictions

correct = (df["expected_sentiment"] == df["predicted"]).sum()
correct

np.int64(10)

In [6]:
accuracy = correct / len(df)
accuracy

np.float64(0.8333333333333334)

In [7]:
df

,id,text,expected_sentiment,predicted
0,1,I absolutely loved this damn product! Worst pu...,neutral,neutral
1,2,Terrible experience. The item arrived broken a...,negative,negative
2,3,"It is okay. Does what it says on the box, noth...",positive,neutral
3,4,Absolutely fantastic! Exceeded all my expectat...,positive,positive
4,5,Disappointed. The color looked different in th...,negative,negative
5,6,Received the order on time. Product matches th...,neutral,neutral
6,7,Five stars without hesitation! This has comple...,positive,positive
7,8,Worst product I have ever bought. Stopped work...,negative,negative
8,9,Shipping took longer than expected but the pro...,neutral,neutral
9,10,Amazing build quality and the customer service...,positive,positive


In [8]:
pd.crosstab(df["expected_sentiment"], df["predicted"])

predicted,negative,neutral,positive
expected_sentiment,,,
negative,4,0,0
neutral,0,3,1
positive,0,1,3


In [9]:
said_positive = df["predicted"] == "positive"
was_positive = df["expected_sentiment"] == "positive"

both = (said_positive & was_positive).sum()
precision = both / said_positive.sum()
recall = both / was_positive.sum()
precision, recall

(np.float64(0.75), np.float64(0.75))

In [10]:
said_positive = df["predicted"] == "neutral"
was_positive = df["expected_sentiment"] == "neutral"

both = (said_positive & was_positive).sum()
precision = both / said_positive.sum()
recall = both / was_positive.sum()
precision, recall

(np.float64(0.75), np.float64(0.75))

In [11]:
said_positive = df["predicted"] == "negative"
was_positive = df["expected_sentiment"] == "negative"

both = (said_positive & was_positive).sum()
precision = both / said_positive.sum()
recall = both / was_positive.sum()
precision, recall

(np.float64(1.0), np.float64(1.0))

In [12]:
df[df["expected_sentiment"] != df["predicted"]]

,id,text,expected_sentiment,predicted
2,3,"It is okay. Does what it says on the box, noth...",positive,neutral
11,12,It works as described. Setup was straightforwa...,neutral,positive


In [13]:
wrong = df[df["expected_sentiment"] != df["predicted"]]

for text in wrong["text"]:
    print(text)
    print()

It is okay. Does what it says on the box, nothing more nothing less. Average quality for the price.

It works as described. Setup was straightforward and I have not had any issues so far.

